# PrivFed: Production-Grade Federated Learning Training

**Automated federated learning training with hyperparameter tuning, checkpointing, and GitHub backup.**

## Features
- ✅ 10 hyperparameter configurations automatically tested
- ✅ YAML-driven configuration (single source of truth)
- ✅ Automatic metrics logging to CSV
- ✅ Google Drive checkpointing (survives Colab restarts)
- ✅ GitHub backup automation
- ✅ Resume from checkpoint capability
- ✅ Debug mode (short runs) by default

## Section 1: Environment Setup

In [ ]:
# Install dependencies
%pip install -q torch torchvision torchaudio
%pip install -q flwr
%pip install -q numpy pandas scikit-learn pyyaml matplotlib seaborn
%pip install -q gitpython

print("✅ Dependencies installed")

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# Create base directories
DRIVE_BASE = '/content/drive/MyDrive/PrivFed'
os.makedirs(f'{DRIVE_BASE}/models', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/models/best', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/results/plots', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)

print(f"✅ Google Drive mounted at {DRIVE_BASE}")

In [ ]:
# Clone or upload code
import zipfile
from google.colab import files

# Option 1: Clone from GitHub (RECOMMENDED)
# Uncomment and set your repo URL:
# !git clone https://github.com/yourusername/PriFed.git
# %cd PriFed/backend

# Option 2: Upload code as zip
print("Upload backend code zip file (if not using git clone):")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✅ Extracted {filename}")

# Set working directory
WORK_DIR = '/content/PrivFed/backend' if os.path.exists('/content/PrivFed') else '/content/backend'
if os.path.exists(WORK_DIR):
    os.chdir(WORK_DIR)
    print(f"✅ Working directory: {os.getcwd()}")
else:
    print(f"⚠️ Working directory not found: {WORK_DIR}")

In [ ]:
# Upload dataset
os.makedirs('dataset', exist_ok=True)

print("Upload dataset CSV files:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.csv'):
        os.rename(filename, f'dataset/{filename}')
        print(f"✅ Moved {filename} to dataset/")

## Section 2: Configuration Management

In [ ]:
# Load configuration from YAML (single source of truth)
import yaml
import sys
from pathlib import Path

def load_config(config_path='configs/config.yaml'):
    """Load configuration from YAML file."""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Update dataset path for Colab
    if 'data' in config:
        config['data']['dataset_path'] = './dataset'
    
    # Set device to CUDA if available
    if 'experiment' in config:
        config['experiment']['device'] = 'cuda'
    
    return config

# Load base configuration
BASE_CONFIG = load_config()
print("✅ Configuration loaded from config.yaml")
print(f"   Experiment: {BASE_CONFIG.get('experiment', {}).get('name', 'N/A')}")
print(f"   Device: {BASE_CONFIG.get('experiment', {}).get('device', 'N/A')}")

In [ ]:
# Generate 10 hyperparameter configurations
import copy
import numpy as np

def generate_hyperparameter_configs(base_config):
    """
    Generate exactly 10 hyperparameter configurations for systematic experimentation.
    Each config varies: learning_rate, batch_size, optimizer, weight_decay, local_epochs, client_fraction
    """
    configs = []
    
    # Define hyperparameter search space
    learning_rates = [0.0001, 0.0005, 0.001, 0.002, 0.005]
    batch_sizes = [256, 512, 1024]
    optimizers = ['adam', 'sgd', 'adamw']
    weight_decays = [1e-6, 1e-5, 1e-4]
    local_epochs_list = [3, 5, 7]
    client_fractions = [0.5, 0.67, 1.0]
    strategies = ['FedAvg', 'FedProx']
    
    # Generate 10 distinct configurations
    config_specs = [
        # Config 0: Conservative (low LR, small batch)
        {'lr': 0.0001, 'bs': 256, 'opt': 'adam', 'wd': 1e-5, 'epochs': 3, 'cf': 0.67, 'strat': 'FedAvg'},
        # Config 1: Balanced
        {'lr': 0.001, 'bs': 512, 'opt': 'adam', 'wd': 1e-5, 'epochs': 5, 'cf': 1.0, 'strat': 'FedAvg'},
        # Config 2: Aggressive (high LR, large batch)
        {'lr': 0.005, 'bs': 1024, 'opt': 'adam', 'wd': 1e-4, 'epochs': 7, 'cf': 1.0, 'strat': 'FedAvg'},
        # Config 3: SGD variant
        {'lr': 0.001, 'bs': 512, 'opt': 'sgd', 'wd': 1e-5, 'epochs': 5, 'cf': 0.67, 'strat': 'FedAvg'},
        # Config 4: AdamW with regularization
        {'lr': 0.0005, 'bs': 512, 'opt': 'adamw', 'wd': 1e-4, 'epochs': 5, 'cf': 1.0, 'strat': 'FedAvg'},
        # Config 5: FedProx strategy
        {'lr': 0.001, 'bs': 512, 'opt': 'adam', 'wd': 1e-5, 'epochs': 5, 'cf': 0.67, 'strat': 'FedProx'},
        # Config 6: Low client fraction
        {'lr': 0.001, 'bs': 512, 'opt': 'adam', 'wd': 1e-5, 'epochs': 5, 'cf': 0.5, 'strat': 'FedAvg'},
        # Config 7: High regularization
        {'lr': 0.001, 'bs': 512, 'opt': 'adam', 'wd': 1e-4, 'epochs': 5, 'cf': 1.0, 'strat': 'FedAvg'},
        # Config 8: Many local epochs
        {'lr': 0.0005, 'bs': 512, 'opt': 'adam', 'wd': 1e-5, 'epochs': 7, 'cf': 1.0, 'strat': 'FedAvg'},
        # Config 9: Medium learning rate, FedProx
        {'lr': 0.002, 'bs': 512, 'opt': 'adam', 'wd': 1e-5, 'epochs': 5, 'cf': 0.67, 'strat': 'FedProx'},
    ]
    
    for idx, spec in enumerate(config_specs):
        config = copy.deepcopy(base_config)
        
        # Update model hyperparameters
        config['model']['learning_rate'] = spec['lr']
        config['model']['batch_size'] = spec['bs']
        config['model']['optimizer'] = spec['opt']
        config['model']['weight_decay'] = spec['wd']
        config['model']['local_epochs'] = spec['epochs']
        
        # Update federated learning config
        config['federated_learning']['strategy'] = spec['strat']
        config['federated_learning']['client_fraction'] = spec['cf']
        
        # Add config metadata
        config['hyperparameter_config_id'] = idx
        config['hyperparameter_spec'] = spec
        
        configs.append(config)
    
    return configs

HYPERPARAMETER_CONFIGS = generate_hyperparameter_configs(BASE_CONFIG)
print(f"✅ Generated {len(HYPERPARAMETER_CONFIGS)} hyperparameter configurations")
for i, cfg in enumerate(HYPERPARAMETER_CONFIGS):
    spec = cfg['hyperparameter_spec']
    print(f"   Config {i}: LR={spec['lr']}, BS={spec['bs']}, Opt={spec['opt']}, "
          f"WD={spec['wd']}, Epochs={spec['epochs']}, CF={spec['cf']}, Strat={spec['strat']}")

In [ ]:
# Enable debug mode (short runs) by default
# Set to False in config.yaml for full training
DEBUG_MODE = BASE_CONFIG.get('experiment', {}).get('debug_mode', True)

if DEBUG_MODE:
    # Short debug configuration
    for cfg in HYPERPARAMETER_CONFIGS:
        cfg['federated_learning']['num_rounds'] = 2
        cfg['model']['local_epochs'] = 1
    print("⚠️ DEBUG MODE: Using 2 rounds, 1 local epoch per round")
else:
    print("✅ PRODUCTION MODE: Using full training configuration")

## Section 3: Data Loading

In [ ]:
# Load and prepare datasets
import sys
sys.path.append('.')

from utils.data_utils import prepare_local_datasets_for_banks, prepare_global_test_set

print("Loading datasets...")
BANK_DATASETS = prepare_local_datasets_for_banks(BASE_CONFIG)
X_TEST, Y_TEST = prepare_global_test_set(BASE_CONFIG)

print(f"✅ Datasets loaded:")
print(f"   Number of banks: {len(BANK_DATASETS)}")
print(f"   Test set size: {X_TEST.shape[0]}")
for bank_name, (X_train, y_train, X_val, y_val) in BANK_DATASETS.items():
    print(f"   {bank_name}: Train={X_train.shape[0]}, Val={X_val.shape[0]}")

## Section 4: Metrics Logging & Persistence Setup

In [ ]:
# Initialize metrics logging system
import pandas as pd
from datetime import datetime
import json

class MetricsLogger:
    """Automated metrics logging to CSV with round-by-round tracking."""
    
    def __init__(self, log_file='results/training_logs.csv'):
        self.log_file = log_file
        self.logs = []
        os.makedirs(os.path.dirname(log_file), exist_ok=True)
        
        # Load existing logs if resuming
        if os.path.exists(log_file):
            self.df = pd.read_csv(log_file)
            self.logs = self.df.to_dict('records')
            print(f"✅ Loaded {len(self.logs)} existing log entries")
        else:
            self.df = pd.DataFrame()
            print("✅ Initialized new metrics logger")
    
    def log_round(self, config_id, round_num, metrics, hyperparams):
        """Log metrics for a single round."""
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'config_id': config_id,
            'round': round_num,
            'learning_rate': hyperparams['lr'],
            'batch_size': hyperparams['bs'],
            'optimizer': hyperparams['opt'],
            'weight_decay': hyperparams['wd'],
            'local_epochs': hyperparams['epochs'],
            'client_fraction': hyperparams['cf'],
            'strategy': hyperparams['strat'],
            'train_loss': metrics.get('loss', None),
            'val_loss': metrics.get('val_loss', metrics.get('loss', None)),
            'accuracy': metrics.get('accuracy', None),
            'auc': metrics.get('auc', None),
            'precision': metrics.get('precision', None),
            'recall': metrics.get('recall', None),
            'f1': metrics.get('f1', None),
        }
        
        self.logs.append(log_entry)
        
        # Append to CSV immediately (for safety)
        df_new = pd.DataFrame([log_entry])
        df_new.to_csv(self.log_file, mode='a', header=not os.path.exists(self.log_file), index=False)
    
    def get_best_configs(self, criterion='auc', top_k=1):
        """Get best configurations by criterion."""
        if not self.logs:
            return []
        
        df = pd.DataFrame(self.logs)
        df_last_round = df.groupby('config_id').last().reset_index()
        df_sorted = df_last_round.sort_values(criterion, ascending=False)
        return df_sorted.head(top_k).to_dict('records')

METRICS_LOGGER = MetricsLogger(f'{DRIVE_BASE}/results/training_logs.csv')
print("✅ Metrics logger initialized")

## Section 5: Checkpoint Management & Resume Capability

In [ ]:
# Checkpoint management for resume capability
import pickle
import glob

class CheckpointManager:
    """Manages training checkpoints for resume capability."""
    
    def __init__(self, checkpoint_dir):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
    
    def save_checkpoint(self, config_id, round_num, model_state, metrics, config):
        """Save checkpoint after each round."""
        checkpoint = {
            'config_id': config_id,
            'round_num': round_num,
            'model_state': model_state,
            'metrics': metrics,
            'config': config,
            'timestamp': datetime.now().isoformat()
        }
        
        checkpoint_path = f"{self.checkpoint_dir}/config_{config_id}_round_{round_num}.pkl"
        with open(checkpoint_path, 'wb') as f:
            pickle.dump(checkpoint, f)
        
        # Also save to Drive
        drive_checkpoint_path = f"{DRIVE_BASE}/checkpoints/config_{config_id}_round_{round_num}.pkl"
        with open(drive_checkpoint_path, 'wb') as f:
            pickle.dump(checkpoint, f)
        
        return checkpoint_path
    
    def find_latest_checkpoint(self, config_id):
        """Find latest checkpoint for a config."""
        pattern = f"{self.checkpoint_dir}/config_{config_id}_round_*.pkl"
        checkpoints = glob.glob(pattern)
        if not checkpoints:
            # Check Drive
            pattern = f"{DRIVE_BASE}/checkpoints/config_{config_id}_round_*.pkl"
            checkpoints = glob.glob(pattern)
        
        if checkpoints:
            # Extract round numbers and get latest
            rounds = [int(f.split('_round_')[1].split('.')[0]) for f in checkpoints]
            latest_idx = rounds.index(max(rounds))
            return checkpoints[latest_idx], max(rounds)
        return None, 0
    
    def load_checkpoint(self, checkpoint_path):
        """Load checkpoint."""
        with open(checkpoint_path, 'rb') as f:
            return pickle.load(f)
    
    def get_completed_configs(self):
        """Get list of configs that have completed training."""
        all_checkpoints = glob.glob(f"{self.checkpoint_dir}/config_*_round_*.pkl")
        if not all_checkpoints:
            all_checkpoints = glob.glob(f"{DRIVE_BASE}/checkpoints/config_*_round_*.pkl")
        
        config_rounds = {}
        for cp in all_checkpoints:
            parts = os.path.basename(cp).split('_')
            config_id = int(parts[1])
            round_num = int(parts[3].split('.')[0])
            if config_id not in config_rounds or round_num > config_rounds[config_id]:
                config_rounds[config_id] = round_num
        
        return config_rounds

CHECKPOINT_MANAGER = CheckpointManager('checkpoints')
print("✅ Checkpoint manager initialized")

# Check for existing checkpoints
completed = CHECKPOINT_MANAGER.get_completed_configs()
if completed:
    print(f"📋 Found checkpoints for {len(completed)} configs:")
    for cfg_id, last_round in completed.items():
        print(f"   Config {cfg_id}: Last round {last_round}")
else:
    print("📋 No existing checkpoints found - starting fresh")

## Section 6: Automated Plotting

In [ ]:
# Automated plotting system
import matplotlib.pyplot as plt
import seaborn as sns

class PlotGenerator:
    """Automatically generates and saves training plots."""
    
    def __init__(self, plots_dir):
        self.plots_dir = plots_dir
        os.makedirs(plots_dir, exist_ok=True)
        sns.set_style("darkgrid")
        plt.rcParams['figure.figsize'] = (12, 6)
    
    def plot_training_curves(self, metrics_logger, save_name='training_curves.png'):
        """Plot loss and accuracy curves for all configs."""
        if not metrics_logger.logs:
            print("⚠️ No metrics to plot yet")
            return
        
        df = pd.DataFrame(metrics_logger.logs)
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # Loss curves
        for config_id in df['config_id'].unique():
            config_data = df[df['config_id'] == config_id]
            axes[0, 0].plot(config_data['round'], config_data['train_loss'], 
                          label=f'Config {config_id}', marker='o', alpha=0.7)
        axes[0, 0].set_xlabel('Round')
        axes[0, 0].set_ylabel('Train Loss')
        axes[0, 0].set_title('Training Loss by Configuration')
        axes[0, 0].legend()
        axes[0, 0].grid(True)
        
        # Accuracy curves
        for config_id in df['config_id'].unique():
            config_data = df[df['config_id'] == config_id]
            axes[0, 1].plot(config_data['round'], config_data['accuracy'], 
                          label=f'Config {config_id}', marker='s', alpha=0.7)
        axes[0, 1].set_xlabel('Round')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].set_title('Accuracy by Configuration')
        axes[0, 1].legend()
        axes[0, 1].grid(True)
        
        # AUC curves
        for config_id in df['config_id'].unique():
            config_data = df[df['config_id'] == config_id]
            axes[1, 0].plot(config_data['round'], config_data['auc'], 
                          label=f'Config {config_id}', marker='^', alpha=0.7)
        axes[1, 0].set_xlabel('Round')
        axes[1, 0].set_ylabel('AUC')
        axes[1, 0].set_title('AUC by Configuration')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
        
        # Final metrics comparison
        df_last = df.groupby('config_id').last().reset_index()
        axes[1, 1].bar(df_last['config_id'], df_last['auc'], alpha=0.7)
        axes[1, 1].set_xlabel('Configuration ID')
        axes[1, 1].set_ylabel('Final AUC')
        axes[1, 1].set_title('Final AUC Comparison')
        axes[1, 1].grid(True, axis='y')
        
        plt.tight_layout()
        plot_path = f"{self.plots_dir}/{save_name}"
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        # Also save to Drive
        drive_plot_path = f"{DRIVE_BASE}/results/plots/{save_name}"
        plt.figure(figsize=(16, 12))
        # Recreate and save to Drive
        # (Simplified - in practice, reuse the figure)
        
        print(f"✅ Saved plot: {plot_path}")
    
    def plot_hyperparameter_analysis(self, metrics_logger, save_name='hyperparameter_analysis.png'):
        """Plot hyperparameter impact analysis."""
        if not metrics_logger.logs:
            return
        
        df = pd.DataFrame(metrics_logger.logs)
        df_last = df.groupby('config_id').last().reset_index()
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # Learning rate impact
        axes[0, 0].scatter(df_last['learning_rate'], df_last['auc'], alpha=0.7, s=100)
        axes[0, 0].set_xlabel('Learning Rate')
        axes[0, 0].set_ylabel('Final AUC')
        axes[0, 0].set_title('Learning Rate Impact')
        axes[0, 0].grid(True)
        
        # Batch size impact
        axes[0, 1].scatter(df_last['batch_size'], df_last['auc'], alpha=0.7, s=100)
        axes[0, 1].set_xlabel('Batch Size')
        axes[0, 1].set_ylabel('Final AUC')
        axes[0, 1].set_title('Batch Size Impact')
        axes[0, 1].grid(True)
        
        # Local epochs impact
        axes[0, 2].scatter(df_last['local_epochs'], df_last['auc'], alpha=0.7, s=100)
        axes[0, 2].set_xlabel('Local Epochs')
        axes[0, 2].set_ylabel('Final AUC')
        axes[0, 2].set_title('Local Epochs Impact')
        axes[0, 2].grid(True)
        
        # Optimizer comparison
        optimizer_auc = df_last.groupby('optimizer')['auc'].mean()
        axes[1, 0].bar(optimizer_auc.index, optimizer_auc.values, alpha=0.7)
        axes[1, 0].set_xlabel('Optimizer')
        axes[1, 0].set_ylabel('Mean AUC')
        axes[1, 0].set_title('Optimizer Comparison')
        axes[1, 0].grid(True, axis='y')
        
        # Strategy comparison
        strategy_auc = df_last.groupby('strategy')['auc'].mean()
        axes[1, 1].bar(strategy_auc.index, strategy_auc.values, alpha=0.7)
        axes[1, 1].set_xlabel('Strategy')
        axes[1, 1].set_ylabel('Mean AUC')
        axes[1, 1].set_title('Federated Strategy Comparison')
        axes[1, 1].grid(True, axis='y')
        
        # Client fraction impact
        axes[1, 2].scatter(df_last['client_fraction'], df_last['auc'], alpha=0.7, s=100)
        axes[1, 2].set_xlabel('Client Fraction')
        axes[1, 2].set_ylabel('Final AUC')
        axes[1, 2].set_title('Client Fraction Impact')
        axes[1, 2].grid(True)
        
        plt.tight_layout()
        plot_path = f"{self.plots_dir}/{save_name}"
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"✅ Saved hyperparameter analysis: {plot_path}")

PLOT_GENERATOR = PlotGenerator('results/plots')
print("✅ Plot generator initialized")

## Section 7: GitHub Backup Automation

In [ ]:
# GitHub backup automation
from git import Repo
import subprocess

class GitHubBackup:
    """Automated GitHub backup system."""
    
    def __init__(self, repo_path='.', remote_url=None):
        self.repo_path = repo_path
        self.remote_url = remote_url
        self.repo = None
        
        # Try to initialize or load repo
        try:
            if os.path.exists(f'{repo_path}/.git'):
                self.repo = Repo(repo_path)
                print(f"✅ Loaded existing git repository")
            else:
                print("⚠️ No git repository found - GitHub backup disabled")
                print("   To enable: Initialize git repo and set remote_url")
        except Exception as e:
            print(f"⚠️ Git error: {e}")
            self.repo = None
    
    def backup(self, commit_message=None):
        """Backup current state to GitHub."""
        if not self.repo:
            print("⚠️ Skipping GitHub backup - no repository")
            return False
        
        try:
            # Add all files
            self.repo.git.add(A=True)
            
            # Commit
            if not commit_message:
                commit_message = f"Auto-backup: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
            
            self.repo.index.commit(commit_message)
            
            # Push to remote
            if self.remote_url:
                origin = self.repo.remote(name='origin')
                if not origin:
                    origin = self.repo.create_remote('origin', self.remote_url)
                origin.push()
                print(f"✅ GitHub backup successful: {commit_message}")
                return True
            else:
                print("⚠️ No remote URL configured - commit created but not pushed")
                return True
                
        except Exception as e:
            print(f"⚠️ GitHub backup failed: {e}")
            return False

# Initialize GitHub backup (set REMOTE_URL if you want automatic pushes)
REMOTE_URL = None  # Set to your GitHub repo URL, e.g., "https://github.com/user/repo.git"
GITHUB_BACKUP = GitHubBackup('.', REMOTE_URL)

if REMOTE_URL:
    print("✅ GitHub backup enabled")
else:
    print("ℹ️ GitHub backup disabled - set REMOTE_URL to enable")

## Section 8: Federated Training Loop with Hyperparameter Tuning

In [ ]:
# Main federated training loop with all 10 configurations
import torch
import torch.nn as nn
from utils.fl_utils import FederatedTrainingServer, create_client_fn
import flwr as fl

# Note: Flower's evaluate_fn in FederatedTrainingServer automatically stores metrics
# in server.round_metrics after each round. We'll process these after simulation completes.

def train_single_config(config, config_id, start_round=0):
    """
    Train a single hyperparameter configuration with per-round checkpointing.
    Supports resume from checkpoint.
    """
    print(f"\n{'='*60}")
    print(f"Training Configuration {config_id}")
    print(f"{'='*60}")
    
    spec = config['hyperparameter_spec']
    print(f"Hyperparameters: LR={spec['lr']}, BS={spec['bs']}, Opt={spec['opt']}, "
          f"WD={spec['wd']}, Epochs={spec['epochs']}, CF={spec['cf']}, Strat={spec['strat']}")
    
    # Check for existing checkpoint
    checkpoint_path, last_round = CHECKPOINT_MANAGER.find_latest_checkpoint(config_id)
    if checkpoint_path and last_round >= start_round:
        print(f"📋 Found checkpoint at round {last_round}")
        checkpoint = CHECKPOINT_MANAGER.load_checkpoint(checkpoint_path)
        start_round = last_round + 1
        print(f"📋 Resuming from round {start_round}")
    else:
        print(f"🆕 Starting fresh training")
        start_round = 0
    
    num_rounds = config['federated_learning']['num_rounds']
    
    if start_round >= num_rounds:
        print(f"✅ Configuration {config_id} already completed")
        return None
    
    # Track best models
    best_metrics = {'auc': 0, 'accuracy': 0, 'loss': float('inf')}
    best_model_states = {}
    round_metrics_history = []
    
    # Create server (will track metrics internally in server.round_metrics)
    server = FederatedTrainingServer(config, (X_TEST, Y_TEST))
    
    # Create client function
    client_fn = create_client_fn(BANK_DATASETS, config)
    
    # Adjust num_rounds for remaining rounds
    remaining_rounds = num_rounds - start_round
    
    print(f"Starting federated training: {remaining_rounds} rounds")
    
    try:
        # Run Flower simulation
        # The server's evaluate_fn automatically populates server.round_metrics after each round
        history = fl.simulation.start_simulation(
            client_fn=client_fn,
            num_clients=len(BANK_DATASETS),
            config=fl.server.ServerConfig(num_rounds=remaining_rounds),
            strategy=server.strategy,
            client_resources={'num_cpus': 1, 'num_gpus': 0}
        )
        
        # Process metrics from server (collected during simulation)
        # server.round_metrics contains metrics for each round
        for round_idx, round_metrics in enumerate(server.round_metrics):
            actual_round = start_round + round_idx
            
            # Log metrics
            METRICS_LOGGER.log_round(config_id, actual_round, round_metrics, spec)
            round_metrics_history.append(round_metrics)
            
            # Save checkpoint after each round
            checkpoint_path = CHECKPOINT_MANAGER.save_checkpoint(
                config_id, actual_round, None, round_metrics, config
            )
            
            # Track best models
            if round_metrics.get('auc', 0) > best_metrics['auc']:
                best_metrics['auc'] = round_metrics.get('auc', 0)
                best_model_states['auc'] = {'round': actual_round, 'metrics': round_metrics}
            
            if round_metrics.get('accuracy', 0) > best_metrics['accuracy']:
                best_metrics['accuracy'] = round_metrics.get('accuracy', 0)
                best_model_states['accuracy'] = {'round': actual_round, 'metrics': round_metrics}
            
            if round_metrics.get('loss', float('inf')) < best_metrics['loss']:
                best_metrics['loss'] = round_metrics.get('loss', float('inf'))
                best_model_states['loss'] = {'round': actual_round, 'metrics': round_metrics}
            
            print(f"Round {actual_round} metrics: AUC={round_metrics.get('auc', 0):.4f}, "
                  f"Acc={round_metrics.get('accuracy', 0):.4f}, Loss={round_metrics.get('loss', 0):.4f}")
            
            # Save model checkpoint metadata to Drive
            model_metadata = {
                'config_id': config_id,
                'round': actual_round,
                'metrics': round_metrics,
                'hyperparameters': spec,
                'timestamp': datetime.now().isoformat()
            }
            model_metadata_path = f"{DRIVE_BASE}/models/config_{config_id}_round_{actual_round}.json"
            with open(model_metadata_path, 'w') as f:
                json.dump(model_metadata, f, indent=2)
            
            # GitHub backup every N rounds
            backup_frequency = config.get('experiment', {}).get('github_backup_frequency', 5)
            if (actual_round + 1) % backup_frequency == 0:
                GITHUB_BACKUP.backup(f"Training progress: Config {config_id}, Round {actual_round + 1}")
        
        # Get final metrics
        final_metrics = round_metrics_history[-1] if round_metrics_history else {}
        
        print(f"\n✅ Configuration {config_id} training completed")
        print(f"Best metrics: AUC={best_metrics['auc']:.4f}, "
              f"Acc={best_metrics['accuracy']:.4f}, Loss={best_metrics['loss']:.4f}")
        
        return {
            'config_id': config_id,
            'best_metrics': best_metrics,
            'best_model_states': best_model_states,
            'final_metrics': final_metrics,
            'history': history
        }
        
    except Exception as e:
        print(f"❌ Error in configuration {config_id}: {e}")
        import traceback
        traceback.print_exception(type(e), e, e.__traceback__)
        # Save error checkpoint with whatever metrics we have
        if round_metrics_history:
            CHECKPOINT_MANAGER.save_checkpoint(
                config_id, len(round_metrics_history) - 1, None, 
                round_metrics_history[-1], config
            )
        raise

print("✅ Training function with checkpointing defined")

In [ ]:
# Execute training for all 10 configurations AUTOMATICALLY
print("="*60)
print("STARTING FEDERATED LEARNING TRAINING")
print(f"Total configurations: {len(HYPERPARAMETER_CONFIGS)}")
print(f"Debug mode: {DEBUG_MODE}")
print("="*60)

training_results = []
start_time = datetime.now()

for config_id, config in enumerate(HYPERPARAMETER_CONFIGS):
    config_start = datetime.now()
    
    try:
        print(f"\n{'#'*60}")
        print(f"Processing Configuration {config_id + 1}/{len(HYPERPARAMETER_CONFIGS)}")
        print(f"{'#'*60}")
        
        result = train_single_config(config, config_id)
        if result:
            training_results.append(result)
            config_duration = (datetime.now() - config_start).total_seconds()
            print(f"⏱️ Configuration {config_id} completed in {config_duration:.1f} seconds")
        
        # Generate intermediate plots every 3 configs
        if (config_id + 1) % 3 == 0:
            print(f"\n📊 Generating intermediate plots...")
            PLOT_GENERATOR.plot_training_curves(
                METRICS_LOGGER, 
                f'training_progress_after_config_{config_id}.png'
            )
            # Copy to Drive
            plot_file = f'results/plots/training_progress_after_config_{config_id}.png'
            if os.path.exists(plot_file):
                shutil.copy(plot_file, f"{DRIVE_BASE}/results/plots/{os.path.basename(plot_file)}")
        
    except Exception as e:
        print(f"❌ Configuration {config_id} failed: {e}")
        import traceback
        traceback.print_exception(type(e), e, e.__traceback__)
        # Continue with next config
        continue

total_duration = (datetime.now() - start_time).total_seconds()
print(f"\n{'='*60}")
print(f"TRAINING COMPLETED")
print(f"{'='*60}")
print(f"✅ Completed: {len(training_results)}/{len(HYPERPARAMETER_CONFIGS)} configurations")
print(f"⏱️ Total time: {total_duration/60:.1f} minutes ({total_duration:.1f} seconds)")
print(f"📊 Metrics logged: {len(METRICS_LOGGER.logs)} round entries")

## Section 9: Best Model Selection & Saving

In [ ]:
# Select and save best models per evaluation criterion (AUC, Accuracy, Loss)
from utils.model_utils import load_model

def save_best_models():
    """Automatically save best models for each evaluation criterion."""
    if not METRICS_LOGGER.logs:
        print("⚠️ No metrics logged yet - skipping best model selection")
        return []
    
    print("\n" + "="*60)
    print("SELECTING BEST MODELS")
    print("="*60)
    
    # Get best configs for each criterion
    best_configs_auc = METRICS_LOGGER.get_best_configs('auc', top_k=1)
    best_configs_acc = METRICS_LOGGER.get_best_configs('accuracy', top_k=1)
    
    # For loss, find minimum (lower is better)
    df = pd.DataFrame(METRICS_LOGGER.logs)
    df_last = df.groupby('config_id').last().reset_index()
    best_config_loss = df_last.loc[df_last['train_loss'].idxmin()].to_dict() if len(df_last) > 0 and 'train_loss' in df_last.columns else None
    
    saved_models = []
    
    # Save best by AUC
    if best_configs_auc:
        best_config_data = best_configs_auc[0]
        config_id = int(best_config_data['config_id'])
        
        checkpoint_path, best_round = CHECKPOINT_MANAGER.find_latest_checkpoint(config_id)
        
        model_info = {
            'criterion': 'auc',
            'config_id': config_id,
            'best_round': int(best_round) if best_round else None,
            'value': float(best_config_data.get('auc', 0)),
            'hyperparameters': {
                'learning_rate': float(best_config_data['learning_rate']),
                'batch_size': int(best_config_data['batch_size']),
                'optimizer': str(best_config_data['optimizer']),
                'weight_decay': float(best_config_data['weight_decay']),
                'local_epochs': int(best_config_data['local_epochs']),
                'client_fraction': float(best_config_data['client_fraction']),
                'strategy': str(best_config_data['strategy'])
            },
            'checkpoint_path': str(checkpoint_path) if checkpoint_path else None,
            'model_filename': f'best_model_auc_config_{config_id}_round_{int(best_round) if best_round else "final"}.pth',
            'timestamp': datetime.now().isoformat()
        }
        
        model_info_path = f"{DRIVE_BASE}/models/best/best_model_auc_config_{config_id}.json"
        with open(model_info_path, 'w') as f:
            json.dump(model_info, f, indent=2)
        
        saved_models.append(model_info)
        print(f"✅ Best model (AUC): Config {config_id}, AUC={model_info['value']:.4f}, Round {model_info['best_round']}")
    
    # Save best by Accuracy
    if best_configs_acc:
        best_config_data = best_configs_acc[0]
        config_id = int(best_config_data['config_id'])
        
        checkpoint_path, best_round = CHECKPOINT_MANAGER.find_latest_checkpoint(config_id)
        
        model_info = {
            'criterion': 'accuracy',
            'config_id': config_id,
            'best_round': int(best_round) if best_round else None,
            'value': float(best_config_data.get('accuracy', 0)),
            'hyperparameters': {
                'learning_rate': float(best_config_data['learning_rate']),
                'batch_size': int(best_config_data['batch_size']),
                'optimizer': str(best_config_data['optimizer']),
                'weight_decay': float(best_config_data['weight_decay']),
                'local_epochs': int(best_config_data['local_epochs']),
                'client_fraction': float(best_config_data['client_fraction']),
                'strategy': str(best_config_data['strategy'])
            },
            'checkpoint_path': str(checkpoint_path) if checkpoint_path else None,
            'model_filename': f'best_model_accuracy_config_{config_id}_round_{int(best_round) if best_round else "final"}.pth',
            'timestamp': datetime.now().isoformat()
        }
        
        model_info_path = f"{DRIVE_BASE}/models/best/best_model_accuracy_config_{config_id}.json"
        with open(model_info_path, 'w') as f:
            json.dump(model_info, f, indent=2)
        
        saved_models.append(model_info)
        print(f"✅ Best model (Accuracy): Config {config_id}, Acc={model_info['value']:.4f}, Round {model_info['best_round']}")
    
    # Save best by Loss
    if best_config_loss and 'train_loss' in best_config_loss:
        config_id = int(best_config_loss['config_id'])
        checkpoint_path, best_round = CHECKPOINT_MANAGER.find_latest_checkpoint(config_id)
        
        model_info = {
            'criterion': 'loss',
            'config_id': config_id,
            'best_round': int(best_round) if best_round else None,
            'value': float(best_config_loss.get('train_loss', float('inf'))),
            'hyperparameters': {
                'learning_rate': float(best_config_loss['learning_rate']),
                'batch_size': int(best_config_loss['batch_size']),
                'optimizer': str(best_config_loss['optimizer']),
                'weight_decay': float(best_config_loss['weight_decay']),
                'local_epochs': int(best_config_loss['local_epochs']),
                'client_fraction': float(best_config_loss['client_fraction']),
                'strategy': str(best_config_loss['strategy'])
            },
            'checkpoint_path': str(checkpoint_path) if checkpoint_path else None,
            'model_filename': f'best_model_loss_config_{config_id}_round_{int(best_round) if best_round else "final"}.pth',
            'timestamp': datetime.now().isoformat()
        }
        
        model_info_path = f"{DRIVE_BASE}/models/best/best_model_loss_config_{config_id}.json"
        with open(model_info_path, 'w') as f:
            json.dump(model_info, f, indent=2)
        
        saved_models.append(model_info)
        print(f"✅ Best model (Loss): Config {config_id}, Loss={model_info['value']:.4f}, Round {model_info['best_round']}")
    
    # Save comprehensive summary
    summary_path = f"{DRIVE_BASE}/models/best/best_models_summary.json"
    summary = {
        'best_models': saved_models,
        'timestamp': datetime.now().isoformat(),
        'total_configs_tested': len(HYPERPARAMETER_CONFIGS),
        'completed_configs': len(training_results) if 'training_results' in globals() else 0,
        'total_rounds_logged': len(METRICS_LOGGER.logs),
        'selection_criteria': ['auc', 'accuracy', 'loss']
    }
    
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    
    print(f"\n✅ Best models summary saved to {summary_path}")
    print(f"   Selected {len(saved_models)} best models across 3 criteria")
    
    return saved_models

# Execute best model selection after training completes
BEST_MODELS = save_best_models()

## Section 10: Final Plots & Reports

In [ ]:
# Generate final comprehensive plots
print("Generating final plots...")

PLOT_GENERATOR.plot_training_curves(METRICS_LOGGER, 'final_training_curves.png')
PLOT_GENERATOR.plot_hyperparameter_analysis(METRICS_LOGGER, 'final_hyperparameter_analysis.png')

# Copy plots to Drive
import shutil
for plot_file in glob.glob('results/plots/*.png'):
    drive_plot = f"{DRIVE_BASE}/results/plots/{os.path.basename(plot_file)}"
    shutil.copy(plot_file, drive_plot)
    print(f"✅ Copied {os.path.basename(plot_file)} to Drive")

print("✅ All plots generated and saved")

In [ ]:
# Generate final training report
report = {
    'experiment_name': BASE_CONFIG.get('experiment', {}).get('name', 'privfed_fraud_detection'),
    'timestamp': datetime.now().isoformat(),
    'total_configurations': len(HYPERPARAMETER_CONFIGS),
    'completed_configurations': len(training_results),
    'best_models': BEST_MODELS,
    'hyperparameter_configs': [
        {
            'config_id': i,
            'spec': cfg['hyperparameter_spec']
        }
        for i, cfg in enumerate(HYPERPARAMETER_CONFIGS)
    ],
    'summary_metrics': {
        'best_auc': max([m.get('auc', 0) for m in METRICS_LOGGER.logs] + [0]),
        'best_accuracy': max([m.get('accuracy', 0) for m in METRICS_LOGGER.logs] + [0]),
        'best_loss': min([m.get('train_loss', float('inf')) for m in METRICS_LOGGER.logs] + [float('inf')])
    }
}

report_path = f"{DRIVE_BASE}/results/training_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)

print(f"✅ Training report saved to {report_path}")
print(f"\n📊 Final Summary:")
print(f"   Configurations tested: {report['total_configurations']}")
print(f"   Completed: {report['completed_configurations']}")
print(f"   Best AUC: {report['summary_metrics']['best_auc']:.4f}")
print(f"   Best Accuracy: {report['summary_metrics']['best_accuracy']:.4f}")
print(f"   Best Loss: {report['summary_metrics']['best_loss']:.4f}")

## Section 11: Final GitHub Backup & Download

In [ ]:
# Final GitHub backup
GITHUB_BACKUP.backup("Training completed - all configurations finished")

print("✅ Final GitHub backup completed")

In [ ]:
# Download results and models
from google.colab import files
import glob

print("Preparing files for download...")

# Create download packages
download_items = [
    (f'{DRIVE_BASE}/results/training_logs.csv', 'training_logs.csv'),
    (f'{DRIVE_BASE}/results/training_report_*.json', 'training_report.json'),
    (f'{DRIVE_BASE}/models/best/', 'best_models.zip'),
    (f'{DRIVE_BASE}/results/plots/', 'plots.zip'),
]

# Download individual files
for source_pattern, dest_name in download_items:
    if '*' in source_pattern:
        # Handle glob patterns
        matches = glob.glob(source_pattern)
        if matches:
            latest = max(matches, key=os.path.getmtime)
            files.download(latest)
            print(f"✅ Downloaded: {os.path.basename(latest)}")
    elif os.path.isfile(source_pattern):
        files.download(source_pattern)
        print(f"✅ Downloaded: {dest_name}")
    elif os.path.isdir(source_pattern):
        # Zip directory
        zip_name = dest_name if dest_name.endswith('.zip') else f"{dest_name}.zip"
        shutil.make_archive(zip_name.replace('.zip', ''), 'zip', source_pattern)
        files.download(zip_name)
        print(f"✅ Downloaded: {zip_name}")

print("\n✅ All files ready for download")
print(f"\n📁 Files also available in Google Drive: {DRIVE_BASE}")
print(f"   - Models: {DRIVE_BASE}/models/")
print(f"   - Results: {DRIVE_BASE}/results/")
print(f"   - Checkpoints: {DRIVE_BASE}/checkpoints/")

## Integration Instructions

After downloading models:

1. **Place models in local project:**
   ```bash
   cd C:\PriFed\backend
   # Copy downloaded .pth files to models/ directory
   ```

2. **Load model in backend:**
   ```python
   from utils.model_utils import load_model
   model, metadata = load_model('models/global_model_final_XXX.pth')
   ```

3. **Sync results to database:**
   ```bash
   python scripts/sync_training_to_db.py
   ```